# 01 -- Build the modelling base table

**What this notebook does (plain English):** This turns the assembled data into
the table every model will use. For each loan it records three things that drive
all later numbers:

- **Default** -- did the loan go badly wrong? (180+ days late, or it ended in a
  loss event such as a foreclosure sale.)
- **EAD (Exposure at Default)** -- how much money was still owed when it defaulted.
- **LGD (Loss Given Default)** -- of that exposure, how much was *actually lost*
  after the property was sold and costs/recoveries settled. This is computed from
  Freddie Mac's **real loss fields**, which is the centrepiece of the project.

**Headline result:** average loss-given-default is far worse in the downturn
(~55-58% in 2007/2008) than in the calm year (~25% in 2015).

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the cached loan-level table from notebook 00.
import numpy as np
import pandas as pd
from src import definitions as d
from src.output import save_csv
df = pd.read_parquet('data/processed/loan_level.parquet')
print(df.shape)

(150000, 54)


In [3]:
# Realised loss and LGD are only defined for defaulted loans that DISPOSED
# (reached a final sale). For everyone else LGD is left blank, on purpose.
df['realised_loss'] = np.where(df['disposed'], d.realised_loss(df), np.nan)
lgd_raw = df['realised_loss'] / df['ead'].replace(0, np.nan)
df['lgd'] = np.where(df['disposed'], d.winsorise_lgd(lgd_raw), np.nan)

In [4]:
# Sanity check: reconcile our computed loss against Freddie Mac's own
# actual_loss_calculation field (their number is stored as a negative loss).
rec = df[df['disposed']].dropna(subset=['actual_loss_calculation'])
rec = rec[rec['actual_loss_calculation'] != 0]
corr = np.corrcoef(-rec['actual_loss_calculation'], d.realised_loss(rec))[0, 1]
print(f'loss reconciliation correlation vs dataset field: {corr:.3f}')

loss reconciliation correlation vs dataset field: 0.990


In [5]:
# Add simple risk bands we will reuse in the EDA and models.
df['credit_score_band'] = pd.cut(df['credit_score'], [0, 620, 660, 700, 740, 780, 851],
                                 right=False, labels=['<620', '620-659', '660-699', '700-739', '740-779', '780+'])
df['ltv_band'] = pd.cut(df['original_ltv'], [0, 60, 70, 80, 90, 200],
                        right=False, labels=['<60', '60-69', '70-79', '80-89', '90+'])

In [6]:
# Keep one clean analysis row per loan and cache it for later notebooks.
base_cols = [
    'loan_sequence_number', 'vintage_year', 'credit_score', 'original_ltv',
    'original_cltv', 'original_dti', 'original_interest_rate', 'original_loan_term',
    'original_upb', 'loan_purpose', 'occupancy_status', 'channel', 'number_of_borrowers',
    'credit_score_band', 'ltv_band', 'ever_default', 'disposed', 'max_delinq_status',
    'ead', 'realised_loss', 'lgd',
]
base = df[base_cols].copy()
base.to_parquet('data/processed/analysis_base.parquet')
print('analysis base:', base.shape)

analysis base: (150000, 21)


In [7]:
# Results table: default rate and average LGD by vintage (downturn vs calm).
tbl = df.groupby('vintage_year').agg(
    loans=('loan_sequence_number', 'size'),
    default_rate=('ever_default', 'mean'),
    disposed_defaults=('disposed', 'sum'),
    avg_lgd=('lgd', 'mean'),
    median_lgd=('lgd', 'median'),
    avg_ead=('ead', 'mean'),
).reset_index().round(4)
save_csv(tbl, 'output/01_default_lgd_by_vintage.csv')
tbl

,vintage_year,loans,default_rate,disposed_defaults,avg_lgd,median_lgd,avg_ead
0,2007,50000,0.1374,4479,0.5783,0.5631,186177.0150
1,2008,50000,0.0735,2134,0.5441,0.5131,198213.9893
2,2015,50000,0.0242,136,0.2464,0.1678,197949.9096


**Reading the table:** both the chance of default *and* the severity of
loss when it happens are much worse in the crisis vintages -- the two effects
compound, which is exactly why a downturn hurts a mortgage book so much.